Практична робота 9
Перетворення даних: похідні змінні, дискретизація, масштабування, індикаторне кодування
Виконавець: Войтович Богдан, IT-32
Варіант: 6 (Чернігів)

In [64]:
import pandas as pd
import numpy as np

# Параметри варіанту
base_price = 690      # базова ціна, грн
base_qty = 4          # базова кількість

# Створення набору з 10 клієнтів
clients = pd.DataFrame({
    "вік": [25, 34, 29, 55, 43, 22, 60, 48, 37, 31],
    "дохід": [12000, 18000, 15000, 25000, 22000, 11000, 30000, 27000, 19000, 16000],
    "кількість": [4, 5, 3, 7, 4, 2, 5, 7, 3, 4],
    "місто_доставки": ["Чернігів", "Львів", "Ужгород", "Чернігів", "Одеса",
                       "Полтава", "Львів", "Київ", "Ужгород", "Чернігів"]
})


In [65]:
# cut — інтервали однакової довжини
clients["вік_cut"] = pd.cut(clients["вік"], bins=3)
print(clients["вік_cut"].value_counts())

# qcut — рівна кількість спостережень у групах
clients["вік_qcut"] = pd.qcut(clients["вік"], q=3)
print(clients["вік_qcut"].value_counts())


вік_cut
(21.962, 34.667]    5
(47.333, 60.0]      3
(34.667, 47.333]    2
Name: count, dtype: int64
вік_qcut
(21.999, 31.0]    4
(43.0, 60.0]      4
(31.0, 43.0]      2
Name: count, dtype: int64


In [66]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

# Масштабуємо вік і дохід окремо
clients_minmax = scaler_minmax.fit_transform(clients[["вік", "дохід"]])
clients_z = scaler_std.fit_transform(clients[["вік", "дохід"]])

df_minmax = pd.DataFrame(clients_minmax, columns=["вік_минмакс", "дохід_минмакс"])
df_z = pd.DataFrame(clients_z, columns=["вік_z", "дохід_z"])

print(pd.concat([df_minmax.head(), df_z.head()], axis=1))


   вік_минмакс  дохід_минмакс     вік_z   дохід_z
0     0.078947       0.052632 -1.105815 -1.238866
1     0.315789       0.368421 -0.363104 -0.247773
2     0.184211       0.210526 -0.775721 -0.743319
3     0.868421       0.736842  1.369891  0.908502
4     0.552632       0.578947  0.379608  0.412955


In [67]:
dummies = pd.get_dummies(clients["місто_доставки"], prefix="місто")
clients = pd.concat([clients, dummies], axis=1)

print(dummies.head())
print(f"Кількість унікальних міст: {clients['місто_доставки'].nunique()}, кількість створених стовпців: {dummies.shape[1]}")


   місто_Київ  місто_Львів  місто_Одеса  місто_Полтава  місто_Ужгород  \
0       False        False        False          False          False   
1       False         True        False          False          False   
2       False        False        False          False           True   
3       False        False        False          False          False   
4       False        False         True          False          False   

   місто_Чернігів  
0            True  
1           False  
2           False  
3            True  
4           False  
Кількість унікальних міст: 6, кількість створених стовпців: 6


Короткі відповіді на контрольні питання
Різниця pd.cut() і pd.qcut():
cut() створює інтервали однакової довжини за шкалою.
qcut() створює інтервали з приблизно однаковою кількістю спостережень.
Чому one-hot для міста?
Місто — категоріальна змінна без порядку, тому один з кращих варіантів — категоріальне (one-hot) кодування, щоб не вносити помилковий порядковий зв’язок.

Навіщо масштабувати?
Методи, які залежать від відстаней (кластеризація, регуляризована регресія), чутливі до масштабу ознак. Масштабування уніфікує значення, даючи змогу коректно порівнювати різні змінні.

Чому частка доходу важливіша?
Вона поєднує інформацію про суму замовлення і доходи клієнта, допомагає виділити, скільки ресурсів витрачає клієнт відносно можливостей, що краще характеризує поведінку.

